# 🐛 Debugging — Part 3: Advanced Techniques

> **Module:** 01 — Programming Best Practices  
> **Series:** Debugging in VS Code (3 of 3)  
> **Covers:** Exception breakpoints, multi-file debugging, test debugging, remote attach, Debug Console, advanced tips

**Previous ←** [02_intermediate.ipynb](02_intermediate.ipynb)

---

## 1 — Debugging Exceptions

VS Code can break **automatically** when an exception is raised — even before a `try/except` catches it.

### Exception breakpoints

In the **BREAKPOINTS** panel (bottom of the Debug sidebar) you'll find checkboxes:

| Option | Behavior |
|--------|----------|
| **Raised Exceptions** | Pause whenever *any* exception is raised, even caught ones |
| **Uncaught Exceptions** | Pause only on exceptions that would crash the program |
| **User Uncaught Exceptions** | Pause on exceptions uncaught in your code but caught by a framework |

### When to use each

- **Uncaught Exceptions** (default): always-on safety net for crashes.
- **Raised Exceptions**: enable temporarily when you need to see the *exact throw point* inside a `try/except`.
- **User Uncaught**: useful inside frameworks (Flask, FastAPI) where the framework wraps your code in its own `try/except`.

### Example

Open `debug_example.py` and look at the Part 3 section. The `parse_config` and `compute_metrics` functions demonstrate both scenarios:

- **`parse_config`**: the JSON error is caught by `try/except`. Enable **"Raised Exceptions"** to pause at `json.loads()` before the except block handles it.
- **`compute_metrics`**: the `ZeroDivisionError` is uncaught. The default **"Uncaught Exceptions"** catches it.

> **Tip:** Enable "Raised Exceptions", find the bug, then disable it — otherwise the debugger stops on every harmless exception.

In [ ]:
# ============================================================
# Exception debugging demo
#
# With "Uncaught Exceptions" enabled → pauses at ZeroDivisionError
# With "Raised Exceptions"  enabled → also pauses at the
#   json.loads() line BEFORE the except block catches it
# ============================================================

import json


def parse_config(raw: str) -> dict:
    try:
        config = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}") from e

    required = ["name", "version", "settings"]
    missing = [k for k in required if k not in config]
    if missing:
        raise ValueError(f"Missing keys: {missing}")
    return config


def compute_metrics(values: list[float]) -> dict:
    total = sum(values)
    count = len([v for v in values if v > 0])  # BUG: ignores zero/negative
    return {"total": total, "count": count, "average": total / count}


# Scenario 1 — caught exception (need "Raised Exceptions" to see throw point)
try:
    parse_config('{"name": "app", version: 1}')  # invalid JSON
except ValueError as e:
    print(f"Handled: {e}")

# Scenario 2 — uncaught exception (debugger always catches this)
compute_metrics([-5, -3, 0, -1])  # ZeroDivisionError

---

## 2 — Debugging Across Multiple Files

Real projects span many modules. The debugger handles this transparently: when you **Step Into** (`F11`) a function call, VS Code follows execution into whatever file that function lives in and opens it automatically.

### Walkthrough

Open `pipeline_example.py` (in this folder) and set a breakpoint on `run_pipeline()`.

1. Press `F11` on `records = load_records(source)` — the editor jumps into `load_records`.
2. Press `Shift+F11` to finish that function and come back.
3. Press `F11` on `clean = process_records(records)` — enters `process_records`, which internally calls `validate_record`.
4. Press `F11` again on `validate_record(record)` to go one level deeper.

The **Call Stack** shows the full chain: `<module>` → `run_pipeline` → `process_records` → `validate_record`.

### The `justMyCode` setting

| Value | Behavior |
|-------|----------|
| `true` (default) | `F11` only enters **your** code. Library internals are skipped. |
| `false` | `F11` enters everything — useful to understand how `pandas.merge()` or `requests.get()` works internally |

Set it in `launch.json`:
```json
"justMyCode": false
```

---

## 3 — Debugging Unit Tests

Debugging a failing test is one of the most common debugging tasks. VS Code integrates directly with pytest and unittest.

### Setup

1. `Ctrl+Shift+P` → **"Python: Configure Tests"** → select **pytest** → select the `tests/` directory
2. Tests appear in the **Testing** sidebar (flask icon, or `Ctrl+Shift+T`)

### Three ways to debug a test

| Method | How |
|--------|-----|
| **Testing sidebar** | Right-click a test → **"Debug Test"** |
| **Inline icons** | Open the test file — click the 🐛 icon above a test function |
| **launch.json** | Use a `"module": "pytest"` config with specific test paths in `"args"` |

### Where to place breakpoints

- **Inside the test** — to see what the test is actually doing
- **Inside the code under test** — to see why it produces the wrong result
- **On the assertion line** — to inspect both actual and expected values right before comparison

> **Tip:** Set `justMyCode: false` so you can step from the test into the production code seamlessly.

In [ ]:
# ============================================================
# Test debugging demo — the function has a subtle bug
# (integer division instead of float division)
# ============================================================

def calculate_discount(price: float, discount_pct: float) -> float:
    if discount_pct < 0 or discount_pct > 100:
        raise ValueError(f"Discount must be 0-100, got {discount_pct}")
    discount_amount = price * discount_pct // 100  # BUG: // truncates
    return price - discount_amount


# --- Tests (would live in tests/test_calculator.py) ---
import pytest  # noqa: E402

def test_no_discount():
    assert calculate_discount(100.0, 0) == 100.0

def test_half_discount():
    assert calculate_discount(100.0, 50) == 50.0  # passes by coincidence

def test_small_discount():
    result = calculate_discount(79.99, 15)
    expected = 67.9915  # 79.99 * 0.15 = 11.9985
    assert result == pytest.approx(expected, abs=0.01)  # FAILS — why?

def test_large_discount():
    assert calculate_discount(200.0, 75) == 50.0  # FAILS — // truncates


# Run manually to see which fail
for fn in [test_no_discount, test_half_discount, test_small_discount, test_large_discount]:
    try:
        fn()
        print(f"  \u2705 {fn.__name__}")
    except Exception as e:
        print(f"  \u274c {fn.__name__}: {e}")

---

## 4 — Remote Debugging & Attach to Process

Sometimes your code is already running (a server, a batch job, a Docker container) and you want to attach the debugger **without restarting**.

### Launch vs. Attach

| Mode | How it works | Use case |
|------|-------------|----------|
| **Launch** | VS Code starts a new process | Scripts, tests — the normal case |
| **Attach** | VS Code connects to a running process | Servers, containers, remote machines |

### Steps

**1. Add `debugpy` to your script:**

```python
import debugpy
debugpy.listen(5678)
print("Waiting for debugger...")
debugpy.wait_for_client()
print("Debugger attached!")
```

**2. Add a `launch.json` attach config:**

```json
{
    "name": "Attach to Process",
    "type": "debugpy",
    "request": "attach",
    "connect": { "host": "localhost", "port": 5678 },
    "justMyCode": false
}
```

**3. Run the script in a separate terminal** (`python my_server.py`). It prints "Waiting for debugger..." and pauses.

**4. In VS Code**, select "Attach to Process" from the debug dropdown and press `F5`. The script resumes, and you can set breakpoints normally.

### Remote machines / containers

Same approach, but change `host` to the remote IP and add `pathMappings` so VS Code maps remote paths to local ones:

```json
{
    "connect": { "host": "192.168.1.100", "port": 5678 },
    "pathMappings": [{
        "localRoot": "${workspaceFolder}/src",
        "remoteRoot": "/app/src"
    }]
}
```

---

## 5 — The Debug Console

The **Debug Console** (`Ctrl+Shift+Y`) is a live Python REPL that runs in the context of your paused program. It's one of the most powerful and underused features.

### What you can do while paused

| Action | Example |
|--------|--------|
| **Evaluate any expression** | `len(records)`, `df.shape`, `sum(r["score"] for r in data)` |
| **Call functions** | `validate_record({"name": "Test", "score": "99"})` |
| **Modify variables** | `threshold = 0.5` — then Continue and the program uses the new value |
| **Import libraries** | `import json; print(json.dumps(config, indent=2))` |
| **Inspect complex objects** | `result.dtypes`, `result.describe()`, `dir(obj)` |

### Example

Open `debug_example.py` and set a breakpoint on `result = analyze_sales(sales)` (Part 3 section). When paused, try in the Debug Console:

```python
len(sales)
[d for d in sales if d["category"] == "electronics"]
sum(d["price"] * d["quantity"] for d in sales)
sales[0]["price"] = 9999.99   # modify, then F5 to continue
```

### Pro tips

- **Test a fix before editing code:** change a variable's value in the console, press `F5`, and see if the program behaves correctly. If it does, apply the fix permanently.
- **Skip loop iterations:** modify a loop counter to jump ahead.
- **Use `repr(obj)`** for detailed output on custom classes.
- **Use `dir(obj)`** to discover available attributes on unfamiliar objects.

In [ ]:
# ============================================================
# Debug Console demo
#
# Set a breakpoint on `result = analyze(data)`, then in the
# Debug Console try:
#   len(data)
#   [d for d in data if d["category"] == "electronics"]
#   sum(d["price"] * d["quantity"] for d in data)
#   data[0]["price"] = 9999.99   ← modify, then Continue
# ============================================================

def analyze(data: list[dict]) -> dict:
    total_revenue = 0.0
    by_category: dict[str, float] = {}

    for item in data:
        revenue = item["price"] * item["quantity"]
        total_revenue += revenue
        cat = item["category"]
        by_category[cat] = by_category.get(cat, 0) + revenue

    return {
        "total_revenue": total_revenue,
        "by_category": by_category,
        "top_category": max(by_category, key=by_category.get),
    }


data = [
    {"product": "Laptop",     "category": "electronics", "price": 999.99, "quantity": 3},
    {"product": "Headphones", "category": "electronics", "price": 149.99, "quantity": 12},
    {"product": "Desk Chair", "category": "furniture",   "price": 299.99, "quantity": 5},
    {"product": "Monitor",    "category": "electronics", "price": 449.99, "quantity": 7},
    {"product": "Notebook",   "category": "stationery",  "price": 12.99,  "quantity": 50},
    {"product": "Bookshelf",  "category": "furniture",   "price": 189.99, "quantity": 2},
]

result = analyze(data)  # \u2190 set breakpoint here
print(f"Total: ${result['total_revenue']:,.2f}")
print(f"Top: {result['top_category']}")

---

## 6 — Advanced Tips

### Environment variables in debug sessions

Inject them via `launch.json` — no need to modify your system:

```json
"env": {
    "DATABASE_URL": "postgresql://localhost/test_db",
    "API_KEY": "test-key-12345",
    "LOG_LEVEL": "DEBUG"
}
```

### Pre-launch tasks

Run a command before the debugger starts (build, lint, start a DB):

```json
"preLaunchTask": "build"
```

Define the task in `.vscode/tasks.json`.

### Compound configurations (multi-target debugging)

Debug two processes simultaneously (e.g., client + server):

```json
"compounds": [{
    "name": "Server + Client",
    "configurations": ["Debug Server", "Debug Client"],
    "stopAll": true
}]
```

### Debugging list comprehensions

List comprehensions execute as a single line, making them hard to debug. Options:

1. Temporarily expand the comprehension into an explicit loop
2. Set a conditional breakpoint with a condition on the iterator variable
3. Use a logpoint: `{[x for x in items if x > threshold]}`

In [ ]:
# ============================================================
# Final challenge — a realistic bug-hunting scenario
#
# This code has 3 bugs. Use everything you've learned to find
# them: breakpoints, step controls, watch expressions, call
# stack, Debug Console.
# ============================================================

from datetime import datetime, timedelta


def days_between(start: str, end: str) -> int:
    """Count business days (Mon-Fri) between two dates."""
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    biz = 0
    cur = s
    while cur <= e:
        if cur.weekday() < 6:   # BUG 1: should be < 5 (Sat=5, Sun=6)
            biz += 1
        cur += timedelta(days=1)
    return biz


def calculate_salary(employees: list[dict]) -> list[dict]:
    results = []
    for emp in employees:
        hours, rate = emp["hours"], emp["rate"]

        if hours > 40:
            regular = 40 * rate
            overtime = (hours - 40) * rate * 1.5
            total = regular + overtime * rate  # BUG 2: extra `* rate`
        else:
            total = hours * rate

        if total > 5000:
            tax = total * 0.30
        if total > 3000:       # BUG 3: should be `elif`
            tax = total * 0.20
        else:
            tax = total * 0.10

        results.append({"name": emp["name"], "gross": round(total, 2), "tax": round(tax, 2)})
    return results


# --- Run ---
print("Business days Mar 2-13 2026:", days_between("2026-03-02", "2026-03-13"))  # expect 10
print()

employees = [
    {"name": "Alice",   "hours": 45, "rate": 50.0},
    {"name": "Bob",     "hours": 38, "rate": 35.0},
    {"name": "Charlie", "hours": 52, "rate": 75.0},
    {"name": "Diana",   "hours": 40, "rate": 60.0},
]

for emp in calculate_salary(employees):
    print(f"  {emp['name']:10s} | Gross: ${emp['gross']:>10,.2f} | Tax: ${emp['tax']:>8,.2f}")

---

## Keyboard Shortcut Cheat Sheet

| Shortcut | Action |
|----------|--------|
| `F5` | Start / Continue |
| `Shift+F5` | Stop |
| `Ctrl+Shift+F5` | Restart |
| `F9` | Toggle breakpoint |
| `F10` | Step Over |
| `F11` | Step Into |
| `Shift+F11` | Step Out |
| `Ctrl+Shift+Y` | Debug Console |
| `Ctrl+Shift+D` | Run & Debug panel |

---

## Series Summary

| Level | Technique |
|-------|----------|
| **Fundamentals** | Breakpoints, Variables panel, Watch, Step Over/Into/Out |
| **Intermediate** | Conditional breakpoints, Logpoints, `launch.json`, CLI args, Call Stack |
| **Advanced** | Exception breakpoints, multi-file stepping, test debugging, remote attach, Debug Console, compound configs |

> **Golden rule:** if you find yourself adding `print()` to understand what your code does, **stop and use the debugger instead.**